In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "dbstorageproyecto")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/FilmDetails.csv"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
film_details_schema = StructType(fields=[

    StructField("id", IntegerType(), False),
    StructField("director", StringType(), True),
    StructField("top_billed", StringType(), True),
    StructField("budget_usd", DoubleType(), True),
    StructField("revenue_usd", DoubleType(), True)
])

In [0]:
df_film_details = spark.read \
    .schema(film_details_schema) \
    .option("multiLine", True) \
    .csv(ruta)

In [0]:
constructors_final_df = df_film_details.select(
    col("id").alias("film_id"),
    col("director"),
    col("top_billed"),
    col("budget_usd"),
    col("revenue_usd")
).withColumn("ingestion_date", current_timestamp())

In [0]:
constructors_final_df.write.option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema}.filmdetails")